# Final Project Evaluation

This notebook answers the project at two levels.

First, it consolidates the technical evidence from Notebooks 04–08: pricing accuracy, exercise decisions, financial consistency, path-based valuation, performance outside the training range, and the six predefined hypotheses.

Second, it answers the practical business question that motivates the project:

> **When does a trained deep-learning surrogate become worth building instead of continuing to use CRR, finite-difference pricing, or QuantLib?**

The answer cannot come from accuracy alone. A neural network is justified only when it is sufficiently accurate inside a controlled domain and its lower marginal runtime repays the cost of generating labels, training, validating, and maintaining the model. The numerical method remains the reference calculation and the fallback outside the validated range.

## 1. Environment and project paths

The notebook can be started from either the repository root or the `notebooks` directory. The output of this phase is written separately under `artifacts/final_evaluation/phase_1_3` so that it cannot be confused with the later final comparison tables.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "notebooks"
    else NOTEBOOK_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.artifact_registry import (
    assert_required_artifacts_valid,
    audit_artifacts,
)
from src.evaluation.final_artifact_adapters import (
    build_package_summary,
    load_all_final_packages,
)
from src.evaluation.final_lineage_audit import (
    assert_phase_1_3_ready,
    audit_package_coherence,
    audit_static_prediction_alignment,
    build_lineage_inventory,
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "final_evaluation"
    / "phase_1_3"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Audit output: {OUTPUT_DIR}")

Project root: D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and_ML_upskill_program\Deep_Learning\deep_learning_american_option_pricing
Audit output: D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and_ML_upskill_program\Deep_Learning\deep_learning_american_option_pricing\artifacts\final_evaluation\phase_1_3


## 2. Phase 1 — current artifact contract

The registry now follows the outputs actually written by Notebooks 04–08. In particular, Notebook 08 is validated against its current two-path boundary table, `median_seconds` runtime output, selection file, final package, canonical checkpoint, and training manifests.

The audit checks presence and basic schema only. A valid file can still contradict another file, so cross-file checks follow in the next section.

In [2]:
artifact_audit = audit_artifacts(PROJECT_ROOT)

display(
    artifact_audit[
        [
            "notebook",
            "category",
            "name",
            "required_for_final",
            "found",
            "valid",
            "rows",
            "notes",
        ]
    ]
)

assert_required_artifacts_valid(artifact_audit)
print("Artifact contract: PASS")

,notebook,category,name,required_for_final,found,valid,rows,notes
0,None,data,production_dataset_manifest,True,True,True,NaN,JSON schema validated
1,04,final_metrics,nb04_final_metrics,True,True,True,NaN,JSON schema validated
2,04,test_predictions,nb04_test_predictions,True,True,True,187811.0,table schema validated
3,04,checkpoint,nb04_checkpoint,True,True,True,NaN,artifact exists and is non-empty
4,04,training_manifest,nb04_training_manifest,True,True,True,NaN,JSON schema validated
5,05,final_metrics,nb05_final_metrics,True,True,True,NaN,JSON schema validated
6,05,test_predictions,nb05_test_predictions,True,True,True,187811.0,table schema validated
7,05,checkpoint,nb05_checkpoint,True,True,True,NaN,artifact exists and is non-empty
8,05,training_manifest,nb05_training_manifest,True,True,True,NaN,JSON schema validated
9,06,final_metrics,nb06_final_metrics,True,True,True,NaN,JSON schema validated


Artifact contract: PASS


## 3. Phase 2 — explicit final-package adapters

Each upstream notebook is loaded through its own contract. The adapter verifies the declared notebook identity, final execution profile, selected model or configuration, canonical checkpoint name, and the existence of its prediction table where applicable.

This prevents Notebook 09 from guessing meanings from the first column of a table or from silently accepting an older package layout.

In [3]:
packages = load_all_final_packages(PROJECT_ROOT)
package_summary = build_package_summary(packages)
package_coherence = audit_package_coherence(packages)

display(package_summary)
display(package_coherence)

invalid_package_checks = package_coherence.loc[
    ~package_coherence["valid"]
]
if not invalid_package_checks.empty:
    display(invalid_package_checks)

print(
    "Package coherence: "
    f"{'PASS' if invalid_package_checks.empty else 'FAIL'}"
)

,notebook,notebook_id,status,training_profile,selected_model,checkpoint,checkpoint_path,benchmark_checkpoint_path,test_prediction_rows,benchmark_test_predictions_path,final_metrics_path
0,04,04_direct_mlp_pricer,complete,full,Direct MLP,best_direct_mlp.pt,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...,None,187811.0,None,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
1,05,05_early_exercise_premium_model,complete,full,Constrained floor residual,best_premium_model.pt,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...,None,187811.0,None,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
2,06,06_exercise_boundary_analysis,complete,full,Multi-task candidate: lambda_1,best_multitask_pricer.pt,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...,None,187811.0,None,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
3,07,07_neural_longstaff_schwartz,complete,final,Classical LSM,neural_lsm_policy.pt,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...,None,NaN,None,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
4,08,08_final_multihead_model,complete,full,Integrated deployment: warm_start,best_integrated_multihead.pt,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...,187811.0,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...


,notebook,check,valid,details
0,04,final_package_complete,True,status='complete'
1,04,canonical_checkpoint_exists,True,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
2,04,checkpoint_name_matches_file,True,declared='best_direct_mlp.pt'; resolved='best_...
3,04,manifest_complete:training_complete.json,True,status='complete'
4,04,manifest_profile_matches:training_complete.json,True,manifest='full'; final='full'
5,04,test_predictions_exist,True,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
6,05,final_package_complete,True,status='complete'
7,05,canonical_checkpoint_exists,True,D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
8,05,checkpoint_name_matches_file,True,declared='best_premium_model.pt'; resolved='be...
9,05,manifest_complete:training_complete.json,True,status='complete'


Package coherence: PASS


## 4. Phase 3 — common static-test alignment

The pricing and exercise results from Notebooks 04, 05, 06, and 08 can be compared directly only if they refer to the same test observations.

Notebook 04 is used as the reference. The audit checks:

- unique `sample_id` values;
- identical sets of test observations;
- identical normalized American-option targets;
- agreement of shared exported contract-state fields.

Notebook 07 is intentionally excluded from this join because it evaluates path-based methods on a separate held-out contract experiment.

In [4]:
(
    static_prediction_alignment,
    static_field_alignment,
) = audit_static_prediction_alignment(packages)

display(static_prediction_alignment)
display(static_field_alignment)

lineage_inventory = build_lineage_inventory(
    PROJECT_ROOT,
    packages,
)
display(lineage_inventory)

,notebook,reference_notebook,observations,reference_observations,duplicate_sample_ids,missing_reference_ids,extra_candidate_ids,same_sample_id_set,target_max_absolute_difference,same_true_target,valid
0,04,04,187811,187811,0,0,0,True,0.000000e+00,True,True
1,05,04,187811,187811,0,0,0,True,0.000000e+00,True,True
2,06,04,187811,187811,0,0,0,True,0.000000e+00,True,True
3,08,04,187811,187811,0,0,0,True,2.980137e-08,True,True
4,08_scratch,04,187811,187811,0,0,0,True,2.980137e-08,True,True


,notebook,field,observations,max_absolute_difference,matches,note
0,04,moneyness,187811,0.0,True,rtol=1e-07; atol=1e-07
1,04,log_moneyness,187811,0.0,True,rtol=1e-07; atol=1e-07
2,04,time_to_maturity,187811,0.0,True,rtol=1e-07; atol=1e-07
3,04,risk_free_rate,187811,0.0,True,rtol=1e-07; atol=1e-07
4,04,dividend_yield,187811,0.0,True,rtol=1e-07; atol=1e-07
5,04,volatility,187811,0.0,True,rtol=1e-07; atol=1e-07
6,05,moneyness,187811,0.0,True,rtol=1e-07; atol=1e-07
7,05,log_moneyness,187811,0.0,True,rtol=1e-07; atol=1e-07
8,05,time_to_maturity,187811,0.0,True,rtol=1e-07; atol=1e-07
9,05,risk_free_rate,187811,0.0,True,rtol=1e-07; atol=1e-07


,notebook,source,dependency_path,file_path,sha256,fingerprint_method,current_production_manifest_semantic_sha256
0,05,training_complete.json,dependencies.feature_scaler,artifacts/direct_mlp/feature_scaler.joblib,58bfc9d12573bbad5230aa16adabe8675c0542026de309...,unspecified,None
1,05,training_complete.json,dependencies.production_manifest,data/manifests/production_dataset_manifest.json,271de662f20e3dbf3fdd674b307ccc3f1aee4508618657...,unspecified,d100b6c1c7c99953deb308039f39798f4a2785d928ae37...
2,06,final_metrics,dependencies.feature_scaler,artifacts/direct_mlp/feature_scaler.joblib,58bfc9d12573bbad5230aa16adabe8675c0542026de309...,unspecified,None
3,06,final_metrics,dependencies.premium_checkpoint,artifacts/premium_models/best_floor_residual_u...,3bea5031ceb0951acafb3d3be3c5559eef4b53ed5e5e1a...,unspecified,None
4,06,final_metrics,dependencies.premium_summary,artifacts/premium_models/evaluation_summary.json,0456b358519dbf4b399b4f79ceb2f5672f3e67e6a3f85b...,unspecified,None
5,06,final_metrics,dependencies.production_manifest,data/manifests/production_dataset_manifest.json,271de662f20e3dbf3fdd674b307ccc3f1aee4508618657...,unspecified,d100b6c1c7c99953deb308039f39798f4a2785d928ae37...
6,06,multitask_training_complete.json,dependencies.feature_scaler,artifacts/direct_mlp/feature_scaler.joblib,58bfc9d12573bbad5230aa16adabe8675c0542026de309...,unspecified,None
7,06,multitask_training_complete.json,dependencies.premium_checkpoint,artifacts/premium_models/best_floor_residual_u...,3bea5031ceb0951acafb3d3be3c5559eef4b53ed5e5e1a...,unspecified,None
8,06,multitask_training_complete.json,dependencies.premium_summary,artifacts/premium_models/evaluation_summary.json,0456b358519dbf4b399b4f79ceb2f5672f3e67e6a3f85b...,unspecified,None
9,06,multitask_training_complete.json,dependencies.production_manifest,data/manifests/production_dataset_manifest.json,271de662f20e3dbf3fdd674b307ccc3f1aee4508618657...,unspecified,d100b6c1c7c99953deb308039f39798f4a2785d928ae37...


## 5. Strict Phase 1–3 gate

The notebook stops here if any package or paired static comparison is inconsistent. Passing this gate does not yet prove any project hypothesis. It only establishes that the evidence layer is safe to use in the next phases.

In [5]:
phase_1_3_results = {
    "package_coherence": package_coherence,
    "static_prediction_alignment": static_prediction_alignment,
    "static_field_alignment": static_field_alignment,
}

assert_phase_1_3_ready(phase_1_3_results)
print("Phase 1–3 readiness gate: PASS")

Phase 1–3 readiness gate: PASS


## 6. Export the audit evidence

These files are diagnostic inputs for the remaining Notebook 09 rebuild. They are not the final project result tables.

In [6]:
phase_outputs = {
    "artifact_audit": artifact_audit,
    "package_summary": package_summary,
    "package_coherence": package_coherence,
    "static_prediction_alignment": static_prediction_alignment,
    "static_field_alignment": static_field_alignment,
    "lineage_inventory": lineage_inventory,
}

exported_paths = {}
for name, table in phase_outputs.items():
    path = OUTPUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    exported_paths[name] = str(path)

pd.Series(exported_paths, name="exported_path")

artifact_audit                 D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
package_summary                D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
package_coherence              D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_prediction_alignment    D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_field_alignment         D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
lineage_inventory              D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
Name: exported_path, dtype: object

## 7. Phase 1–3 checkpoint

A successful run confirms that the upstream evidence can be compared safely. Phase 4 now uses the verified prediction packages to build one common static-test matrix. Notebook 07 remains outside this matrix because its path-based experiment uses a separate held-out contract sample.


## 8. Phase 4 — authoritative common static comparison

All static pricing outputs are now joined on `sample_id` and evaluated through one implementation. This avoids differences caused by separate summary formats or slightly different metric code in the upstream notebooks.

The comparison includes the analytical proxy, direct neural baseline, premium-model candidates, Notebook 06 price outputs, and both Notebook 08 price outputs. The fixed strike of 100 is read from the production manifest, allowing normalized errors to be translated into option-price units without assuming a value inside the evaluation code.


In [7]:
from src.evaluation.final_static_comparison import (
    assert_phase_4_ready,
    build_static_model_registry,
    run_phase_4_static_comparison,
)

PHASE_4_OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "final_evaluation"
    / "phase_4"
)
PHASE_4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

static_model_registry = build_static_model_registry()
phase_4_results = run_phase_4_static_comparison(
    PROJECT_ROOT,
    packages,
)

static_prediction_matrix = phase_4_results[
    "static_prediction_matrix"
]
static_model_metrics = phase_4_results[
    "static_model_metrics"
]
static_financial_consistency = phase_4_results[
    "static_financial_consistency"
]
static_pairwise_error_comparison = phase_4_results[
    "static_pairwise_error_comparison"
]
static_segmented_pricing = phase_4_results[
    "static_segmented_pricing"
]
static_boundary_pricing = phase_4_results[
    "static_boundary_pricing"
]

print(
    "Aligned static test observations: "
    f"{len(static_prediction_matrix):,}"
)
display(static_model_registry)

Aligned static test observations: 187,811


,model_id,model,source_notebook,prediction_column,evaluation_role,source_selected,financially_constrained
0,black_scholes_proxy,Black–Scholes proxy,04,normalized_european_price,analytical proxy,False,False
1,direct_mlp,Direct MLP,04,direct_mlp_prediction,direct neural baseline,True,False
2,zero_premium_baseline,Zero-premium baseline,05,zero_premium_baseline,naive residual baseline,False,False
3,mean_premium_baseline,Mean-premium baseline,05,mean_premium_baseline,naive residual baseline,False,False
4,unconstrained_premium_mlp,Unconstrained premium MLP,05,unconstrained_premium,residual candidate,False,False
5,nonnegative_premium_mlp,Non-negative premium MLP,05,nonnegative_premium,residual candidate,False,False
6,constrained_floor_residual_mlp,Constrained floor residual MLP,05,constrained_floor_prediction,selected static price model,True,True
7,price_only_constrained_residual_mlp,Price-only constrained residual MLP,06,price_only_normalized_price,price-only control,False,True
8,multitask_constrained_residual_mlp,Multi-task constrained residual MLP,06,predicted_normalized_american_price,selected joint price-decision model,True,True
9,integrated_warm_start_constrained_price,Integrated warm-start constrained price,08,predicted_normalized_american_price,preferred in-domain integrated deployment output,True,True


## 9. Recomputed pricing and financial-consistency results

The first table ranks models only by normalized mean absolute error on the common test sample. The second table reports lower-bound violations from the same rows. These are technical evidence tables; the final task-specific model recommendation will be made only after the exercise, path-based, out-of-domain, and runtime sections are also complete.


In [8]:
pricing_columns = [
    "pricing_rank",
    "model",
    "source_notebook",
    "evaluation_role",
    "source_selected",
    "normalized_mae",
    "normalized_rmse",
    "normalized_median_absolute_error",
    "normalized_max_absolute_error",
    "price_mae",
    "price_rmse",
]
display(static_model_metrics[pricing_columns])

consistency_columns = [
    "model",
    "source_notebook",
    "financially_constrained",
    "negative_count",
    "below_european_count",
    "below_intrinsic_count",
    "below_financial_floor_count",
    "below_financial_floor_rate",
]
display(
    static_financial_consistency[
        consistency_columns
    ]
)

,pricing_rank,model,source_notebook,evaluation_role,source_selected,normalized_mae,normalized_rmse,normalized_median_absolute_error,normalized_max_absolute_error,price_mae,price_rmse
0,1,Constrained floor residual MLP,05,selected static price model,True,0.000102,0.000213,0.000046,0.004641,0.010187,0.021275
1,2,Price-only constrained residual MLP,06,price-only control,False,0.000102,0.000213,0.000046,0.004641,0.010187,0.021275
2,3,Multi-task constrained residual MLP,06,selected joint price-decision model,True,0.000199,0.000405,0.000073,0.007345,0.019891,0.040510
3,4,Non-negative premium MLP,05,residual candidate,False,0.000205,0.000356,0.000115,0.006268,0.020505,0.035591
4,5,Unconstrained premium MLP,05,residual candidate,False,0.000220,0.000359,0.000142,0.007582,0.021956,0.035919
5,6,Integrated warm-start constrained price,08,preferred in-domain integrated deployment output,True,0.000294,0.000589,0.000091,0.009025,0.029391,0.058867
6,7,Integrated balanced-scratch constrained price,08_scratch,scratch experiment winner and robustness bench...,False,0.000423,0.000910,0.000094,0.010462,0.042254,0.090952
7,8,Direct MLP,04,direct neural baseline,True,0.000782,0.001146,0.000548,0.019201,0.078167,0.114556
8,9,Integrated warm-start direct price head,08,auxiliary warm-start integrated output,False,0.002963,0.003965,0.002267,0.038683,0.296283,0.396501
9,10,Integrated balanced-scratch direct price head,08_scratch,auxiliary scratch benchmark output,False,0.005576,0.007443,0.004375,0.051736,0.557584,0.744268


,model,source_notebook,financially_constrained,negative_count,below_european_count,below_intrinsic_count,below_financial_floor_count,below_financial_floor_rate
0,Constrained floor residual MLP,05,True,0,0,0,0,0.000000
1,Integrated balanced-scratch constrained price,08_scratch,True,0,0,0,0,0.000000
2,Integrated warm-start constrained price,08,True,0,0,0,0,0.000000
3,Multi-task constrained residual MLP,06,True,0,0,0,0,0.000000
4,Price-only constrained residual MLP,06,True,0,0,0,0,0.000000
5,Non-negative premium MLP,05,False,0,0,23706,23706,0.126223
6,Mean-premium baseline,05,False,0,0,38540,38540,0.205206
7,Unconstrained premium MLP,05,False,2725,25955,31513,54420,0.289759
8,Integrated warm-start direct price head,08,False,0,34404,27121,57008,0.303539
9,Black–Scholes proxy,04,False,0,0,57780,57780,0.307650


## 10. Paired, segmented, and boundary evidence

Aggregate error can hide where one model improves or deteriorates. The paired table compares absolute errors observation by observation. The segmented table repeats the comparison by moneyness, maturity, volatility, and exercise region. The boundary table focuses on progressively wider bands around the exercise-versus-continuation transition.


In [9]:
direct_pairwise = static_pairwise_error_comparison.loc[
    static_pairwise_error_comparison[
        ["model_a_id", "model_b_id"]
    ].eq("direct_mlp").any(axis=1)
]
display(direct_pairwise)

segment_leaders = (
    static_segmented_pricing
    .sort_values(
        [
            "segment_type",
            "segment",
            "normalized_mae",
        ]
    )
    .groupby(
        ["segment_type", "segment"],
        observed=True,
        group_keys=False,
    )
    .head(3)
)
display(segment_leaders)

boundary_leaders = (
    static_boundary_pricing
    .sort_values(
        ["boundary_limit", "normalized_mae"]
    )
    .groupby(
        "boundary_limit",
        group_keys=False,
    )
    .head(5)
)
display(boundary_leaders)

,model_a_id,model_a,model_b_id,model_b,observations,mean_absolute_error_difference_a_minus_b,median_absolute_error_difference_a_minus_b,model_a_win_rate,model_b_win_rate,tie_rate,model_a_lower_mean_absolute_error
0,black_scholes_proxy,Black–Scholes proxy,direct_mlp,Direct MLP,187811,0.012617,0.002146,0.333814,0.666186,0.000000,False
12,direct_mlp,Direct MLP,zero_premium_baseline,Zero-premium baseline,187811,-0.012617,-0.002146,0.666186,0.333814,0.000000,True
13,direct_mlp,Direct MLP,mean_premium_baseline,Mean-premium baseline,187811,-0.015139,-0.011793,0.983872,0.016128,0.000000,True
14,direct_mlp,Direct MLP,unconstrained_premium_mlp,Unconstrained premium MLP,187811,0.000562,0.000375,0.171822,0.828178,0.000000,False
15,direct_mlp,Direct MLP,nonnegative_premium_mlp,Non-negative premium MLP,187811,0.000577,0.000387,0.151839,0.848161,0.000000,False
16,direct_mlp,Direct MLP,constrained_floor_residual_mlp,Constrained floor residual MLP,187811,0.000680,0.000460,0.084750,0.915250,0.000000,False
17,direct_mlp,Direct MLP,price_only_constrained_residual_mlp,Price-only constrained residual MLP,187811,0.000680,0.000460,0.084750,0.915250,0.000000,False
18,direct_mlp,Direct MLP,multitask_constrained_residual_mlp,Multi-task constrained residual MLP,187811,0.000583,0.000390,0.159969,0.840015,0.000016,False
19,direct_mlp,Direct MLP,integrated_warm_start_constrained_price,Integrated warm-start constrained price,187811,0.000488,0.000340,0.221238,0.778756,0.000005,False
20,direct_mlp,Direct MLP,integrated_warm_start_direct_price_head,Integrated warm-start direct price head,187811,-0.002181,-0.001603,0.853752,0.146248,0.000000,True


,segment_type,segment,model_id,model,source_notebook,observations,normalized_mae,normalized_rmse,price_mae,price_rmse
0,exercise_region,continue,constrained_floor_residual_mlp,Constrained floor residual MLP,05,142651,0.000124,0.000238,0.012415,0.023786
1,exercise_region,continue,price_only_constrained_residual_mlp,Price-only constrained residual MLP,06,142651,0.000124,0.000238,0.012415,0.023786
2,exercise_region,continue,nonnegative_premium_mlp,Non-negative premium MLP,05,142651,0.000151,0.000265,0.015110,0.026536
13,exercise_region,exercise,integrated_scratch_constrained_price,Integrated balanced-scratch constrained price,08_scratch,45160,0.000007,0.000043,0.000685,0.004329
14,exercise_region,exercise,multitask_constrained_residual_mlp,Multi-task constrained residual MLP,06,45160,0.000028,0.000110,0.002829,0.011047
15,exercise_region,exercise,constrained_floor_residual_mlp,Constrained floor residual MLP,05,45160,0.000031,0.000098,0.003148,0.009755
26,maturity,long,constrained_floor_residual_mlp,Constrained floor residual MLP,05,85885,0.000137,0.000264,0.013678,0.026362
27,maturity,long,price_only_constrained_residual_mlp,Price-only constrained residual MLP,06,85885,0.000137,0.000264,0.013678,0.026362
28,maturity,long,nonnegative_premium_mlp,Non-negative premium MLP,05,85885,0.000231,0.000391,0.023109,0.039094
39,maturity,medium,constrained_floor_residual_mlp,Constrained floor residual MLP,05,50120,0.000086,0.000176,0.008598,0.017561


,boundary_limit,boundary_band,model_id,model,source_notebook,observations,exercise_observations,continuation_observations,normalized_mae,normalized_rmse,price_mae,price_rmse
0,0.001,≤0.001,integrated_scratch_constrained_price,Integrated balanced-scratch constrained price,08_scratch,59673,45160,14513,0.000023,0.000086,0.002259,0.008562
1,0.001,≤0.001,constrained_floor_residual_mlp,Constrained floor residual MLP,05,59673,45160,14513,0.000052,0.000138,0.005249,0.013836
2,0.001,≤0.001,price_only_constrained_residual_mlp,Price-only constrained residual MLP,06,59673,45160,14513,0.000052,0.000138,0.005249,0.013836
3,0.001,≤0.001,multitask_constrained_residual_mlp,Multi-task constrained residual MLP,06,59673,45160,14513,0.000053,0.000161,0.005306,0.016124
4,0.001,≤0.001,integrated_warm_start_constrained_price,Integrated warm-start constrained price,08,59673,45160,14513,0.000068,0.000198,0.006796,0.019767
13,0.005,≤0.005,integrated_scratch_constrained_price,Integrated balanced-scratch constrained price,08_scratch,73673,45160,28513,0.000066,0.000202,0.006615,0.020182
14,0.005,≤0.005,constrained_floor_residual_mlp,Constrained floor residual MLP,05,73673,45160,28513,0.000073,0.000186,0.007320,0.018633
15,0.005,≤0.005,price_only_constrained_residual_mlp,Price-only constrained residual MLP,06,73673,45160,28513,0.000073,0.000186,0.007320,0.018633
16,0.005,≤0.005,multitask_constrained_residual_mlp,Multi-task constrained residual MLP,06,73673,45160,28513,0.000099,0.000267,0.009935,0.026725
17,0.005,≤0.005,integrated_warm_start_constrained_price,Integrated warm-start constrained price,08,73673,45160,28513,0.000132,0.000346,0.013233,0.034627


## 11. Strict Phase 4 gate

The notebook stops if any model prediction is missing or non-finite, if the model registry is incomplete, if duplicate sample identifiers appear, or if any required comparison table is empty. Passing this gate establishes the common static pricing evidence needed by the later hypothesis and conclusion phases.


In [10]:
assert_phase_4_ready(phase_4_results)
print("Phase 4 readiness gate: PASS")

Phase 4 readiness gate: PASS


## 12. Export Phase 4 evidence

The aligned prediction matrix is stored as Parquet because it contains one row per test observation. The smaller evidence tables are stored as CSV. These remain intermediate final-evaluation inputs rather than the completed project report.


In [11]:
phase_4_export_paths = {}

matrix_path = (
    PHASE_4_OUTPUT_DIR
    / "static_prediction_matrix.parquet"
)
static_prediction_matrix.to_parquet(
    matrix_path,
    index=False,
)
phase_4_export_paths[
    "static_prediction_matrix"
] = str(matrix_path)

phase_4_tables = {
    "static_model_registry": static_model_registry,
    "static_model_metrics": static_model_metrics,
    "static_financial_consistency": (
        static_financial_consistency
    ),
    "static_pairwise_error_comparison": (
        static_pairwise_error_comparison
    ),
    "static_segmented_pricing": (
        static_segmented_pricing
    ),
    "static_boundary_pricing": (
        static_boundary_pricing
    ),
}

for name, table in phase_4_tables.items():
    path = PHASE_4_OUTPUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    phase_4_export_paths[name] = str(path)

pd.Series(
    phase_4_export_paths,
    name="exported_path",
)


static_prediction_matrix            D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_model_registry               D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_model_metrics                D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_financial_consistency        D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_pairwise_error_comparison    D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_segmented_pricing            D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_boundary_pricing             D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
Name: exported_path, dtype: object

## 13. Technical checkpoint before Phase 5

At this point Notebook 09 has one audited and reproducible static pricing comparison. The next phase will keep the remaining experiment families separate: exercise decisions from Notebooks 06 and 08, followed by the path-based Longstaff–Schwartz results from Notebook 07 and the distinct runtime categories.


## 14. Phases 5–6 — remaining evidence families and hypothesis decisions

Phase 4 established the authoritative static pricing comparison. The next step keeps the other tasks separate rather than forcing them into the same ranking:

- exercise decisions from Notebooks 06 and 08 are recomputed on their common test observations;
- classical and neural Longstaff–Schwartz remain on their separate contract-level experiment;
- out-of-domain pricing is compared model by model across the four predefined regimes;
- runtime is separated into static inference, numerical valuation, path-based valuation, simulation, and one-time training;
- H1–H6 are then decided from these validated tables under the predefined thresholds.

No model is trained or loaded in this phase.

In [12]:
from src.evaluation.final_phase_5_6 import (
    assert_phases_5_6_ready,
    run_phases_5_6,
)

PHASE_5_6_OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "final_evaluation"
    / "phase_5_6"
)
PHASE_5_6_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

phase_5_6_results = run_phases_5_6(
    packages,
    static_model_metrics=static_model_metrics,
    static_financial_consistency=(
        static_financial_consistency
    ),
)

exercise_model_registry = phase_5_6_results[
    "exercise_model_registry"
]
exercise_prediction_matrix = phase_5_6_results[
    "exercise_prediction_matrix"
]
exercise_model_metrics = phase_5_6_results[
    "exercise_model_metrics"
]
exercise_boundary_metrics = phase_5_6_results[
    "exercise_boundary_metrics"
]
exercise_ood_comparison = phase_5_6_results[
    "exercise_ood_comparison"
]

static_ood_comparison = phase_5_6_results[
    "static_ood_comparison"
]
static_ood_model_summary = phase_5_6_results[
    "static_ood_model_summary"
]
runtime_comparison = phase_5_6_results[
    "runtime_comparison"
]

hypothesis_evidence = phase_5_6_results[
    "hypothesis_evidence"
]
hypothesis_evidence_table = phase_5_6_results[
    "hypothesis_evidence_table"
]
hypothesis_decisions = phase_5_6_results[
    "hypothesis_decisions"
]

print(
    "Aligned exercise observations: "
    f"{len(exercise_prediction_matrix):,}"
)

Aligned exercise observations: 187,811


## 15. Exercise decisions

The four decision paths answer related but different questions:

1. the specialist classifier from Notebook 06;
2. the exercise output of the Notebook 06 multi-task model;
3. the direct exercise output of Notebook 08;
4. the exercise decision inferred from Notebook 08's estimate of the value of waiting.

The first table uses the full common test set. The second focuses on cumulative bands around the exercise boundary. The third shows how each path behaves in the four out-of-domain regimes.

In [13]:
exercise_columns = [
    "exercise_rank",
    "model",
    "source_notebook",
    "evaluation_role",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "brier_score",
    "roc_auc",
    "pr_auc",
]
display(exercise_model_metrics[exercise_columns])

boundary_columns = [
    "boundary_band",
    "model",
    "observations",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "f1",
    "decision_errors",
    "normalized_mean_regret_when_wrong",
    "normalized_total_regret",
]
display(
    exercise_boundary_metrics[
        boundary_columns
    ]
)

display(exercise_ood_comparison)

,exercise_rank,model,source_notebook,evaluation_role,threshold,accuracy,balanced_accuracy,precision,recall,f1,brier_score,roc_auc,pr_auc
0,1,Integrated warm-start exercise head,08,preferred in-domain combined deployment path,0.690,0.998365,0.997925,0.996129,0.997077,0.996603,0.003751,0.999984,0.999951
1,2,Exercise-only classifier,06,specialist exercise model,0.670,0.998264,0.997874,0.995666,0.997121,0.996393,0.003694,0.999982,0.999946
2,3,Integrated balanced-scratch exercise head,08_scratch,controlled scratch robustness benchmark,0.715,0.998136,0.997328,0.996477,0.995771,0.996124,0.004788,0.999976,0.999925
3,4,Multi-task exercise head,06,joint price-decision specialist,0.675,0.997710,0.997040,0.994735,0.995748,0.995242,0.004010,0.999974,0.999918
4,5,Integrated warm-start continuation-implied dec...,08,decision inferred from deployment continuation...,0.665,0.993025,0.986972,0.995592,0.975310,0.985347,0.006929,0.999815,0.999424
5,6,Integrated balanced-scratch continuation-impli...,08_scratch,scratch continuation benchmark,0.760,0.992844,0.986716,0.995230,0.974911,0.984966,0.007359,0.999753,0.999215


,boundary_band,model,observations,threshold,accuracy,balanced_accuracy,f1,decision_errors,normalized_mean_regret_when_wrong,normalized_total_regret
0,≤0.001,Integrated warm-start exercise head,59673,0.690,0.994855,0.992509,0.996603,307,0.000015,0.004553
1,≤0.001,Exercise-only classifier,59673,0.670,0.994537,0.991808,0.996393,326,0.000012,0.003972
2,≤0.001,Integrated balanced-scratch exercise head,59673,0.715,0.994135,0.992407,0.996124,350,0.000011,0.003953
3,≤0.001,Multi-task exercise head,59673,0.675,0.992794,0.989675,0.995242,430,0.000021,0.009118
4,≤0.001,Integrated warm-start continuation-implied dec...,59673,0.665,0.978047,0.980937,0.985347,1310,0.000011,0.013990
5,≤0.001,Integrated balanced-scratch continuation-impli...,59673,0.760,0.977477,0.980186,0.984966,1344,0.000013,0.017458
6,≤0.005,Integrated warm-start exercise head,73673,0.690,0.995833,0.995470,0.996603,307,0.000015,0.004553
7,≤0.005,Exercise-only classifier,73673,0.670,0.995575,0.995124,0.996393,326,0.000012,0.003972
8,≤0.005,Integrated balanced-scratch exercise head,73673,0.715,0.995249,0.995097,0.996124,350,0.000011,0.003953
9,≤0.005,Multi-task exercise head,73673,0.675,0.994163,0.993701,0.995242,430,0.000021,0.009118


,ood_set,model_id,model,source_notebook,observations,positive_rate,threshold,accuracy,balanced_accuracy,precision,recall,f1,brier_score,roc_auc,pr_auc,mean_regret_all,mean_regret_when_wrong,maximum_regret,total_regret
0,extreme_moneyness,exercise_only_classifier,Exercise-only classifier,06,50000.0,0.36468,0.670,0.98390,0.980648,0.986980,0.968630,0.977719,0.010650,0.999135,0.998471,NaN,NaN,NaN,NaN
1,extreme_moneyness,integrated_scratch_continuation_path,Integrated balanced-scratch continuation-impli...,08_scratch,50000.0,NaN,NaN,0.94222,0.933080,0.939660,0.899309,0.919042,NaN,NaN,NaN,6.266230e-03,0.108450,3.762533,313.311510
2,extreme_moneyness,integrated_scratch_exercise_head,Integrated balanced-scratch exercise head,08_scratch,50000.0,NaN,NaN,0.97426,0.967606,0.985782,0.943019,0.963926,NaN,NaN,NaN,4.866657e-04,0.018907,1.145859,24.333286
3,extreme_moneyness,integrated_warm_start_continuation_path,Integrated warm-start continuation-implied dec...,08,50000.0,NaN,NaN,0.93190,0.924783,0.913363,0.898486,0.905864,NaN,NaN,NaN,1.254350e-02,0.184192,6.341885,627.175103
4,extreme_moneyness,integrated_warm_start_exercise_head,Integrated warm-start exercise head,08,50000.0,NaN,NaN,0.98432,0.980020,0.992660,0.964133,0.978188,NaN,NaN,NaN,1.680977e-04,0.010721,0.336552,8.404883
5,extreme_moneyness,multitask_exercise_head,Multi-task exercise head,06,50000.0,0.36468,0.675,0.98658,0.984264,0.987347,0.975705,0.981491,0.009756,0.999230,0.998641,NaN,NaN,NaN,NaN
6,high_volatility,exercise_only_classifier,Exercise-only classifier,06,50000.0,0.00244,0.670,0.99820,0.982744,0.578431,0.967213,0.723926,0.001715,0.999554,0.827105,NaN,NaN,NaN,NaN
7,high_volatility,integrated_scratch_continuation_path,Integrated balanced-scratch continuation-impli...,08_scratch,50000.0,NaN,NaN,0.99932,0.942422,0.843750,0.885246,0.864000,NaN,NaN,NaN,2.139132e-06,0.003146,0.035351,0.106957
8,high_volatility,integrated_scratch_exercise_head,Integrated balanced-scratch exercise head,08_scratch,50000.0,NaN,NaN,0.99904,0.811455,0.974359,0.622951,0.760000,NaN,NaN,NaN,4.580184e-07,0.000477,0.001737,0.022901
9,high_volatility,integrated_warm_start_continuation_path,Integrated warm-start continuation-implied dec...,08,50000.0,NaN,NaN,0.97708,0.955805,0.091054,0.934426,0.165939,NaN,NaN,NaN,2.302001e-01,10.043633,29.602320,11510.003023


## 16. Out-of-domain pricing, path-based methods, and runtime

The out-of-domain table compares each static pricing model with its own in-domain error. This prevents a model with a larger starting error from appearing robust merely because its absolute deterioration is smaller.

The Longstaff–Schwartz tables remain separate. They are based on held-out contracts and simulated paths, not the 187,811 static test observations.

The runtime table also keeps different kinds of work separate. The H5 comparison uses the selected static pricing model and high-resolution CRR valuation. Neural Longstaff–Schwartz timing is reported as a different path-based workflow.

In [14]:
display(
    static_ood_model_summary[
        [
            "model",
            "source_notebook",
            "h6_eligible",
            "regimes",
            "in_domain_normalized_mae",
            "aggregate_ood_normalized_mae",
            "aggregate_ood_to_in_domain_ratio",
            "minimum_regime_ratio",
            "maximum_regime_ratio",
            "regimes_with_material_deterioration",
        ]
    ]
)

display(static_ood_comparison)

for table_name in (
    "lsm_heldout_pricing",
    "lsm_ood_pricing",
    "lsm_coverage",
    "lsm_financial_bounds",
    "lsm_policy_summary",
):
    table = phase_5_6_results[table_name]
    if not table.empty:
        print(table_name)
        display(table)

runtime_columns = [
    "benchmark_family",
    "method",
    "source_notebook",
    "observations_per_measurement",
    "benchmark_repetitions",
    "median_seconds",
    "seconds_per_observation",
    "observations_per_second",
    "device",
    "cost_frequency",
]
display(runtime_comparison[runtime_columns])

,model,source_notebook,h6_eligible,regimes,in_domain_normalized_mae,aggregate_ood_normalized_mae,aggregate_ood_to_in_domain_ratio,minimum_regime_ratio,maximum_regime_ratio,regimes_with_material_deterioration
0,Multi-task constrained residual MLP,06,True,4,0.000199,0.005683,28.572565,2.397177,52.407965,4
1,Direct MLP,04,True,4,0.000782,0.019690,25.189107,11.796701,58.881738,4
2,Integrated warm-start constrained price,08,True,4,0.000294,0.005617,19.113150,2.607948,49.416263,4
3,Non-negative premium MLP,05,True,4,0.000205,0.003641,17.755295,2.479858,45.760606,4
4,Constrained floor residual MLP,05,True,4,0.000102,0.001770,17.373774,1.158667,33.693760,3
5,Price-only constrained residual MLP,06,True,4,0.000102,0.001770,17.373774,1.158667,33.693760,3
6,Integrated balanced-scratch constrained price,08_scratch,True,4,0.000423,0.003434,8.127309,0.994779,17.128979,3
7,Integrated warm-start direct price head,08,False,4,0.002963,0.033062,11.158784,3.965684,22.207456,4
8,Integrated balanced-scratch direct price head,08_scratch,False,4,0.005576,0.018069,3.240596,1.491933,4.601600,4


,ood_set,model_id,model,source_notebook,observations,ood_normalized_mae,ood_normalized_rmse,h6_eligible,in_domain_normalized_mae,ood_to_in_domain_mae_ratio,relative_ood_deterioration
0,extreme_moneyness,constrained_floor_residual_mlp,Constrained floor residual MLP,05,50000.0,0.000118,0.000378,True,0.000102,1.158667,0.158667
1,high_volatility,constrained_floor_residual_mlp,Constrained floor residual MLP,05,50000.0,0.000568,0.001110,True,0.000102,5.575363,4.575363
2,long_maturity,constrained_floor_residual_mlp,Constrained floor residual MLP,05,50000.0,0.003432,0.007965,True,0.000102,33.693760,32.693760
3,rate_dividend,constrained_floor_residual_mlp,Constrained floor residual MLP,05,50000.0,0.002961,0.006051,True,0.000102,29.067303,28.067303
4,extreme_moneyness,direct_mlp,Direct MLP,04,50000.0,0.011993,0.020319,True,0.000782,15.342281,14.342281
5,high_volatility,direct_mlp,Direct MLP,04,50000.0,0.009221,0.014600,True,0.000782,11.796701,10.796701
6,long_maturity,direct_mlp,Direct MLP,04,50000.0,0.046026,0.087061,True,0.000782,58.881738,57.881738
7,rate_dividend,direct_mlp,Direct MLP,04,50000.0,0.011519,0.017558,True,0.000782,14.735709,13.735709
8,extreme_moneyness,integrated_scratch_constrained_price,Integrated balanced-scratch constrained price,08_scratch,50000.0,0.000420,0.001403,True,0.000423,0.994779,-0.005221
9,high_volatility,integrated_scratch_constrained_price,Integrated balanced-scratch constrained price,08_scratch,50000.0,0.003619,0.006261,True,0.000423,8.565323,7.565323


lsm_heldout_pricing


,source_notebook,mae,maximum_absolute_error,mean_bias,median_absolute_error,method,normalized_mae,rmse
0,07,0.039594,0.127381,-0.033838,0.032319,classical_lsm_price,0.004014,0.052043
1,07,0.087197,0.400425,-0.084348,0.063737,neural_lsm_price,0.006911,0.121771


lsm_ood_pricing


...

lsm_financial_bounds


,source_notebook,check,maximum_violation,mean_positive_violation,method,violation_rate,violations
0,07,negative_price,0.000000e+00,0.000000e+00,classical_lsm_price,0.00,0
1,07,below_intrinsic,1.421085e-14,1.065814e-14,classical_lsm_price,0.04,4
2,07,below_european,7.367725e-02,2.309322e-02,classical_lsm_price,0.23,23
3,07,negative_price,0.000000e+00,0.000000e+00,neural_lsm_price,0.00,0
4,07,below_intrinsic,1.421085e-14,8.881784e-15,neural_lsm_price,0.05,5
5,07,below_european,3.999538e-01,1.147124e-01,neural_lsm_price,0.31,31


lsm_policy_summary


,source_notebook,25%,50%,75%,count,max,mean,metric,min,std
0,07,0.537000,0.688110,0.823110,100.0,1.000000,0.658710,exact_step_agreement,0.000000,0.246566
1,07,0.390540,0.863020,1.585565,100.0,12.303340,1.414097,mean_absolute_step_error,0.000000,1.937613
2,07,0.891620,0.945200,0.969000,100.0,1.000000,0.887792,early_exercise_agreement,0.102480,0.172282
3,07,836.500000,2245.000000,5025.500000,100.0,44876.000000,5109.080000,false_early_exercise_count,0.000000,8724.523158
4,07,0.030360,0.090860,0.271058,100.0,1.000000,0.188116,false_early_exercise_rate,0.000000,0.235879
5,07,37.250000,208.500000,455.250000,100.0,9211.000000,501.320000,missed_early_exercise_count,0.000000,1106.694321
6,07,0.002762,0.011227,0.034936,100.0,0.942013,0.049191,missed_early_exercise_rate,0.000000,0.133510
7,07,0.131780,0.224860,0.366055,100.0,1.000000,0.267091,stopping_distribution_tv,0.000000,0.219760
8,07,0.000383,0.024376,0.061940,100.0,0.390507,0.050510,mean_discounted_payoff_difference,-0.028605,0.080918
9,07,0.390400,0.800138,1.245133,100.0,3.897485,0.932019,mean_neural_shortfall_vs_classical,0.000000,0.754459


,benchmark_family,method,source_notebook,observations_per_measurement,benchmark_repetitions,median_seconds,seconds_per_observation,observations_per_second,device,cost_frequency
0,numerical valuation,High-resolution CRR,07,1.0,100.0,0.548987,0.548987,1.821538,cpu,repeated per-contract valuation
1,path-based valuation,Classical LSM fit and valuation,07,1.0,100.0,0.248967,0.248967,4.016596,cpu,repeated per-contract valuation
2,path-based valuation,Classical LSM end-to-end,07,1.0,100.0,0.382622,0.382622,2.613543,cpu,repeated per-contract valuation
3,path-based valuation,Neural LSM evaluation,07,1.0,100.0,0.581623,0.581623,1.719327,cpu,repeated per-contract valuation
4,path-based valuation,Neural LSM end-to-end,07,1.0,100.0,0.677521,0.677521,1.475969,cpu,repeated per-contract valuation
5,simulation component,Policy path simulation,07,1.0,100.0,0.049365,0.049365,20.257144,cpu,repeated per-contract valuation
6,simulation component,Valuation path simulation,07,1.0,100.0,0.096133,0.096133,10.402261,cpu,repeated per-contract valuation
7,static neural inference,Exercise-only classifier,06,100000.0,NaN,0.181505,0.000002,550947.947122,None,repeated marginal inference
8,static neural inference,Non-negative premium MLP,05,100000.0,NaN,0.196972,0.000002,507687.145047,cpu,repeated marginal inference
9,static neural inference,Price-only constrained residual MLP,06,100000.0,NaN,0.200708,0.000002,498235.622977,None,repeated marginal inference


## 17. H1–H6 decisions

The decisions below are generated from the exact evidence inventory shown first. The rules are fixed in code and are not changed after seeing the results.

H4 uses Notebook 06 as the primary experiment because that notebook directly compares the specialist and multi-task designs. Notebook 08 is included as supporting evidence about the later integrated model.

H6 is evaluated across every eligible static neural pricing model with complete results in all four predefined out-of-domain regimes.

In [15]:
display(hypothesis_evidence_table)
display(hypothesis_decisions)

,evidence_key,value
0,allowed_f1_degradation,0.001
1,black_scholes_mae,0.013398
2,classifier_boundary_f1,0.996393
3,constrained_violation_rate,0.0
4,crr_seconds_per_option,0.548987
5,direct_mlp_mae,0.000782
6,direct_violation_rate,0.308129
7,h5_static_model,Constrained floor residual
8,h6_eligible_models,7
9,h6_maximum_aggregate_ratio,28.572565


,hypothesis,decision,primary_evidence,secondary_evidence,threshold,limitation
0,H1,Supported,Common-test MAE ratio = 0.058341.,Direct MLP MAE=0.00078167348; Black–Scholes pr...,Supported requires a ratio no greater than 0.95.,The result concerns the fixed synthetic in-dom...
1,H2,Supported,Selected residual/direct MAE ratio = 0.130322.,Selected residual=Constrained floor residual; ...,Supported requires a ratio no greater than 0.98.,The comparison uses the same aligned test obse...
2,H3,Supported,Direct violation rate=0.30812892; constrained ...,Rates are recomputed from the common Phase 4 p...,Supported requires zero constrained violations...,"The decision covers non-negativity, European, ..."
3,H4,Not supported,Notebook 06 multi-task F1 change=-0.001152; re...,Notebook 08 integrated exercise F1=0.996603; s...,Allowed F1 degradation=0.001000; required boun...,Exercise labels and boundary distances are gen...
4,H5,Supported,Selected static neural / CRR marginal-runtime ...,Static model=Constrained floor residual; stati...,Supported requires a ratio no greater than 0.50.,"Batch size, hardware, and implementation affec..."
5,H6,Supported,Eligible models with aggregate ratio >=1.25: 7...,multitask_constrained_residual_mlp=28.5726; di...,Supported requires every eligible model's aggr...,The decision applies to the four predefined sy...


## 18. Strict Phases 5–6 gate

The notebook stops if any exercise path is missing, if an eligible model lacks one of the four out-of-domain regimes, if the selected static pricer or CRR runtime is unavailable, or if any hypothesis remains inconclusive.

In [16]:
assert_phases_5_6_ready(
    phase_5_6_results
)
print("Phases 5–6 readiness gate: PASS")

Phases 5–6 readiness gate: PASS


## 19. Export Phases 5–6 evidence

The row-level exercise matrix is written as Parquet. Tables are written as CSV, and the exact hypothesis evidence mapping is written as JSON. These files are technical inputs for the final presentation and reporting work in Phases 7–8.

In [17]:
import json
import math
from collections.abc import Mapping

import numpy as np


def _json_safe(value):
    if isinstance(value, Mapping):
        return {
            str(key): _json_safe(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


phase_5_6_export_paths = {}

exercise_matrix_path = (
    PHASE_5_6_OUTPUT_DIR
    / "exercise_prediction_matrix.parquet"
)
exercise_prediction_matrix.to_parquet(
    exercise_matrix_path,
    index=False,
)
phase_5_6_export_paths[
    "exercise_prediction_matrix"
] = str(exercise_matrix_path)

phase_5_6_tables = {
    name: table
    for name, table in phase_5_6_results.items()
    if isinstance(table, pd.DataFrame)
    and name != "exercise_prediction_matrix"
}

for name, table in phase_5_6_tables.items():
    path = PHASE_5_6_OUTPUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    phase_5_6_export_paths[name] = str(path)

evidence_path = (
    PHASE_5_6_OUTPUT_DIR
    / "hypothesis_evidence.json"
)
evidence_path.write_text(
    json.dumps(
        _json_safe(hypothesis_evidence),
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
phase_5_6_export_paths[
    "hypothesis_evidence"
] = str(evidence_path)

pd.Series(
    phase_5_6_export_paths,
    name="exported_path",
)

exercise_prediction_matrix    D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
exercise_model_registry       D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
exercise_model_metrics        D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
exercise_boundary_metrics     D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
exercise_ood_comparison       D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_ood_comparison         D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
static_ood_model_summary      D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
runtime_comparison            D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
lsm_heldout_pricing           D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
lsm_ood_pricing               D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
lsm_financial_bounds          D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
lsm_policy_summary            D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and...
lsm_runtime_source            D:\Users\k

## 20. Transition to the final conclusion

Phases 1–6 have established the evidence. The remaining work is interpretation and final validation.

The results already show that the project does not have one model that is best at everything:

- Notebook 05 provides the strongest static price;
- Notebook 06 provides the strongest specialist exercise decision;
- Notebook 08 provides the strongest one-model combination of price and exercise information;
- classical Longstaff–Schwartz remains stronger than the neural path-based policy.

Phase 7 turns those results into a complete project conclusion. Phase 8 writes the final package and verifies that every required table, chart, decision, and summary is present and non-empty.


## 21. The central business question

The previous sections establish that the neural models can learn the American put pricing rule. That is not enough to justify deployment.

The standard numerical methods already work. They are flexible, transparent, and require no training. A surrogate makes practical sense only when repeated valuation volume is high enough to offset:

- production of numerical training labels;
- model training and validation;
- model loading and preprocessing;
- the loss of flexibility outside the learned contract domain.

This section measures three different thresholds:

1. **Marginal speed:** which method is faster per valuation at scale?
2. **Operational crossover:** how many contracts must be priced in one job before neural inference is faster?
3. **Lifecycle break-even:** how many cumulative valuations are required to recover the upfront neural build cost?

The benchmark uses the preferred Notebook 05 price model, the preferred Notebook 08 warm-start combined model, the project’s high-resolution Numba CRR, and optional QuantLib binomial and finite-difference engines when QuantLib is installed.

In [18]:
from src.evaluation.business_case_benchmark import (
    RuntimeScalingConfig,
)
from src.evaluation.final_business_case import (
    run_final_business_case,
)

BUSINESS_CASE_OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "final_evaluation"
    / "business_case"
)
BUSINESS_CASE_CHART_DIR = (
    BUSINESS_CASE_OUTPUT_DIR
    / "charts"
)
BUSINESS_CASE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BUSINESS_CASE_DEVICE = "cpu"
INCLUDE_QUANTLIB = True

# Exact numerical work is capped where necessary. Larger requested workloads
# are extrapolated only from the explicitly recorded measured basis.
business_benchmark_config = RuntimeScalingConfig(
    batch_sizes=(
        1,
        10,
        100,
        1_000,
        10_000,
        100_000,
        1_000_000,
    ),
    repeats=5,
    warmup_runs=1,
    cold_repeats=1,
    optional_repeats=3,
    seed=42,
    strike=100.0,
    crr_steps=250,
    project_crr_exact_limit=100_000,
    quantlib_exact_limit=1_000,
    accuracy_sample_size=250,
)

# Historical manifests are used wherever they contain wall-clock times.
# Missing components remain visible. The deployment-preparation value is set
# to zero because this project does not measure a separate packaging process.
UPFRONT_TIME_OVERRIDES_SECONDS = {
    "deployment_preparation": 0.0,
}

business_case_results = run_final_business_case(
    PROJECT_ROOT,
    static_model_metrics=static_model_metrics,
    output_dir=BUSINESS_CASE_CHART_DIR,
    config=business_benchmark_config,
    device=BUSINESS_CASE_DEVICE,
    include_quantlib=INCLUDE_QUANTLIB,
    overrides_seconds=(
        UPFRONT_TIME_OVERRIDES_SECONDS
    ),
)

runtime_scaling = business_case_results[
    "runtime_scaling"
]
accuracy_speed_tradeoff = business_case_results[
    "accuracy_speed_tradeoff"
]
runtime_environment = business_case_results[
    "runtime_environment"
]
runtime_curves = business_case_results[
    "runtime_curves"
]
operational_crossover = business_case_results[
    "operational_crossover"
]
upfront_cost_inventory = business_case_results[
    "upfront_cost_inventory"
]
upfront_cost_scenarios = business_case_results[
    "upfront_cost_scenarios"
]
lifecycle_break_even = business_case_results[
    "lifecycle_break_even"
]
business_case_scenarios = business_case_results[
    "business_case_scenarios"
]
business_case_recommendations = (
    business_case_results[
        "business_case_recommendations"
    ]
)
research_question_7_summary = (
    business_case_results[
        "research_question_7_summary"
    ]
)
business_case_markdown = business_case_results[
    "business_case_markdown"
]
business_case_chart_paths = (
    business_case_results[
        "business_case_chart_paths"
    ]
)
business_case_readiness_audit = (
    business_case_results[
        "business_case_readiness_audit"
    ]
)

print(
    "QuantLib available: "
    f"{runtime_environment['quantlib_available']}"
)
print(
    "Benchmark device: "
    f"{runtime_environment['torch_device']}"
)

C:\Users\kamen.dimitrov\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\kamen.dimitrov\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\kamen.dimitrov\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.htm

QuantLib available: True
Benchmark device: cpu


## 22. Accuracy before speed

The neural methods are compared with the project CRR on the same deterministic in-domain contracts. Runtime is useful only when the resulting approximation remains fit for the intended task.

Notebook 05 returns price only. Notebook 08 returns a protected price and an exercise recommendation. QuantLib results are included only when the package is available in the active environment.

In [19]:
accuracy_speed_columns = [
    "method",
    "family",
    "output_scope",
    "price_mae",
    "price_rmse",
    "maximum_absolute_error",
    "warm_seconds_per_observation_at_scale",
    "warm_observations_per_second_at_scale",
    "status",
]

missing_accuracy_speed_columns = [
    column
    for column in accuracy_speed_columns
    if column not in accuracy_speed_tradeoff.columns
]
if missing_accuracy_speed_columns:
    raise KeyError(
        "accuracy_speed_tradeoff is missing required columns: "
        f"{missing_accuracy_speed_columns}. "
        f"Available columns: {accuracy_speed_tradeoff.columns.tolist()}"
    )

display(
    accuracy_speed_tradeoff.loc[
        :,
        accuracy_speed_columns,
    ]
)

failed_optional_methods = runtime_scaling.loc[
    runtime_scaling["status"].astype(str).eq(
        "failed"
    )
]
if not failed_optional_methods.empty:
    print(
        "Optional benchmark methods that could "
        "not be executed:"
    )
    display(
        failed_optional_methods[
            [
                "method",
                "timing_mode",
                "requested_observations",
                "notes",
            ]
        ].drop_duplicates()
    )

,method,family,output_scope,price_mae,price_rmse,maximum_absolute_error,warm_seconds_per_observation_at_scale,warm_observations_per_second_at_scale,status
0,Project high-resolution Numba CRR,numerical valuation,price and root exercise decision,0.000000,0.000000,0.000000,0.000018,54079.024129,complete
1,QuantLib binomial CRR,numerical valuation,price only,0.004993,0.009545,0.077978,0.001275,784.551128,complete
2,QuantLib finite-difference American put,numerical valuation,price only,0.009126,0.013654,0.050183,0.011687,85.565558,complete
3,Notebook 05 constrained residual model,static neural inference,price only,0.010187,0.021275,0.464150,0.000002,409479.995078,complete
4,Notebook 08 warm-start integrated model,static neural inference,price and exercise decision,0.029391,0.058867,0.902514,0.000003,329054.704976,complete


## 23. Runtime scaling and operational crossover

Cold timing includes model or engine setup. Warm timing represents a running valuation service in which the method is already loaded.

The operational crossover is the smallest workload at which the fitted neural total-time curve falls below the numerical curve. Measured and extrapolated benchmark rows are labelled explicitly.

In [20]:
warm_runtime_view = runtime_scaling.loc[
    runtime_scaling[
        "timing_mode"
    ].astype(str).eq("warm")
]

display(
    warm_runtime_view[
        [
            "method",
            "requested_observations",
            "basis_observations",
            "measurement_type",
            "median_seconds",
            "seconds_per_observation",
            "observations_per_second",
            "status",
        ]
    ]
)

display(
    operational_crossover[
        [
            "timing_mode",
            "neural_method",
            "numerical_method",
            "curve_crossover_observations",
            "smallest_measured_neural_win",
            "status",
        ]
    ]
)

,method,requested_observations,basis_observations,measurement_type,median_seconds,seconds_per_observation,observations_per_second,status
7,Notebook 05 constrained residual model,1,1,measured,0.005322,0.005322,187.913409,complete
8,Notebook 05 constrained residual model,10,10,measured,0.006180,0.000618,1617.992095,complete
9,Notebook 05 constrained residual model,100,100,measured,0.005718,0.000057,17489.550000,complete
10,Notebook 05 constrained residual model,1000,1000,measured,0.008574,0.000009,116638.477806,complete
11,Notebook 05 constrained residual model,10000,10000,measured,0.026111,0.000003,382983.285149,complete
12,Notebook 05 constrained residual model,100000,100000,measured,0.240161,0.000002,416387.340270,complete
13,Notebook 05 constrained residual model,1000000,1000000,measured,2.445253,0.000002,408955.604268,complete
21,Notebook 08 warm-start integrated model,1,1,measured,0.007284,0.007284,137.287204,complete
22,Notebook 08 warm-start integrated model,10,10,measured,0.007604,0.000760,1315.062721,complete
23,Notebook 08 warm-start integrated model,100,100,measured,0.007367,0.000074,13574.599244,complete


KeyError: "['smallest_measured_neural_win'] not in index"

## 24. Upfront cost and lifecycle break-even

The lifecycle calculation includes the cost of producing numerical labels and training the selected deployment model.

Where saved manifests contain wall-clock time, the notebook uses it. Where historical time was not retained, the notebook does not fabricate a single estimate. It reports explicit total-build scenarios so the reader can see how the conclusion changes under one, four, eight, twenty-four, and forty-eight hours of upfront work.

In [ ]:
display(upfront_cost_inventory)
display(upfront_cost_scenarios)

display(
    lifecycle_break_even[
        [
            "neural_method",
            "numerical_method",
            "scenario_id",
            "evidence_type",
            "upfront_hours",
            "marginal_seconds_saved_per_valuation",
            "break_even_valuations",
            "status",
        ]
    ]
)

## 25. Business workloads

A break-even count is easier to understand when translated into actual work.

The scenarios distinguish small one-off pricing from daily portfolio valuation and large risk or stress grids. The same model may be unjustified for one hundred prices and compelling for ten million repeated valuations.

In [ ]:
scenario_columns = [
    "scenario",
    "valuations_per_run",
    "runs_per_year",
    "neural_method",
    "numerical_method",
    "numerical_seconds_per_run",
    "neural_seconds_per_run",
    "seconds_saved_per_run",
    "annual_hours_saved",
    "upfront_scenario_id",
    "payback_runs",
    "payback_years",
    "neural_faster_for_workload",
]
display(
    business_case_scenarios[
        scenario_columns
    ]
)
display(business_case_recommendations)

## 26. Answer to Research Question 7

Research Question 7 is answered at three levels rather than by one headline speed ratio:

- marginal speed at scale;
- crossover within one valuation job;
- cumulative break-even after upfront model-building cost.

This distinction prevents the project from claiming that deep learning is automatically useful merely because inference is fast.

In [ ]:
display(
    pd.Series(
        research_question_7_summary,
        name="Research Question 7",
    )
)
display(Markdown(business_case_markdown))

for chart_name, chart_path in (
    business_case_chart_paths.items()
):
    print(chart_name)
    display(
        Image(filename=str(chart_path))
    )

## 27. Strict business-case gate

The practical conclusion is allowed to enter the final report only when the benchmark contains the three required core methods, both cold and warm timing, an accuracy check, runtime curves, operational crossover, lifecycle scenarios, business workloads, and the four business-case charts.

In [ ]:
display(business_case_readiness_audit)

if not business_case_readiness_audit[
    "valid"
].astype(bool).all():
    raise RuntimeError(
        "Business-case readiness gate failed."
    )

print("Business-case readiness gate: PASS")

## 28. Export business-case evidence

In [ ]:
business_case_tables = {
    "runtime_scaling": runtime_scaling,
    "accuracy_speed_tradeoff": (
        accuracy_speed_tradeoff
    ),
    "runtime_curves": runtime_curves,
    "operational_crossover": (
        operational_crossover
    ),
    "upfront_cost_inventory": (
        upfront_cost_inventory
    ),
    "upfront_cost_scenarios": (
        upfront_cost_scenarios
    ),
    "lifecycle_break_even": (
        lifecycle_break_even
    ),
    "business_case_scenarios": (
        business_case_scenarios
    ),
    "business_case_recommendations": (
        business_case_recommendations
    ),
    "business_case_readiness_audit": (
        business_case_readiness_audit
    ),
}

business_case_export_paths = {}
for name, table in business_case_tables.items():
    path = (
        BUSINESS_CASE_OUTPUT_DIR
        / f"{name}.csv"
    )
    table.to_csv(path, index=False)
    business_case_export_paths[name] = str(path)

runtime_environment_path = (
    BUSINESS_CASE_OUTPUT_DIR
    / "runtime_environment.json"
)
runtime_environment_path.write_text(
    json.dumps(
        _json_safe(runtime_environment),
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
business_case_export_paths[
    "runtime_environment"
] = str(runtime_environment_path)

rq7_path = (
    BUSINESS_CASE_OUTPUT_DIR
    / "research_question_7_summary.json"
)
rq7_path.write_text(
    json.dumps(
        _json_safe(
            research_question_7_summary
        ),
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
business_case_export_paths[
    "research_question_7_summary"
] = str(rq7_path)

pd.Series(
    business_case_export_paths,
    name="exported_path",
)

## 29. Phase 7 — final charts

The charts below show only comparisons that are genuinely like for like. Static prices use the common aligned test set. Exercise decisions use their common decision sample. Longstaff–Schwartz remains on its separate held-out contract experiment. Runtime uses a logarithmic scale because the numerical and neural timings differ by several orders of magnitude.


In [ ]:
from IPython.display import Image, Markdown

from src.evaluation.final_charts import (
    generate_final_charts,
)

FINAL_OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "final_evaluation"
    / "final"
)
FINAL_CHART_DIR = FINAL_OUTPUT_DIR / "charts"

final_chart_paths = generate_final_charts(
    FINAL_CHART_DIR,
    static_model_metrics=static_model_metrics,
    exercise_model_metrics=exercise_model_metrics,
    static_ood_model_summary=(
        static_ood_model_summary
    ),
    runtime_comparison=runtime_comparison,
    lsm_heldout_pricing=phase_5_6_results[
        "lsm_heldout_pricing"
    ],
)

# The business-case charts are part of the final evidence package.
final_chart_paths.update(
    business_case_chart_paths
)

chart_titles = {
    "static_pricing_mae": "Static pricing accuracy",
    "exercise_f1": "Exercise-decision accuracy",
    "ood_deterioration": "Out-of-domain deterioration",
    "runtime_comparison": "Static neural inference versus CRR",
    "lsm_heldout_mae": "Path-based pricing",
    "business_runtime_scaling": "Business-case runtime scaling",
    "business_speedup_vs_crr": "Neural speedup versus CRR",
    "business_lifecycle_break_even": "Lifecycle break-even",
    "business_workload_scenarios": "Business workload scenarios",
}

for chart_name, chart_path in final_chart_paths.items():
    print(chart_titles.get(chart_name, chart_name))
    display(Image(filename=str(chart_path)))


## 30. Task-specific recommendations and the role of Notebook 08

The recommendation table answers practical questions rather than forcing every model into one ranking.

The integrated model is examined separately because its purpose is different. It was built to return several related answers from one shared representation. The relevant question is therefore not whether it wins every individual metric, but how much accuracy and speed are exchanged for that combined output.


In [ ]:
from src.evaluation.final_phase_7_8 import (
    run_phase_7_8_pre_export,
)

phase_7_8_results = run_phase_7_8_pre_export(
    artifact_audit=artifact_audit,
    package_coherence=package_coherence,
    static_prediction_alignment=(
        static_prediction_alignment
    ),
    static_field_alignment=static_field_alignment,
    static_model_metrics=static_model_metrics,
    static_financial_consistency=(
        static_financial_consistency
    ),
    exercise_model_metrics=exercise_model_metrics,
    exercise_boundary_metrics=(
        exercise_boundary_metrics
    ),
    static_ood_model_summary=(
        static_ood_model_summary
    ),
    lsm_heldout_pricing=phase_5_6_results[
        "lsm_heldout_pricing"
    ],
    lsm_coverage=phase_5_6_results[
        "lsm_coverage"
    ],
    runtime_comparison=runtime_comparison,
    hypothesis_decisions=hypothesis_decisions,
    chart_paths=final_chart_paths,
)

task_recommendations = phase_7_8_results[
    "task_recommendations"
]
integrated_model_tradeoff = phase_7_8_results[
    "integrated_model_tradeoff"
]
project_findings = phase_7_8_results[
    "project_findings"
]
project_limitations = phase_7_8_results[
    "project_limitations"
]
final_results_summary = phase_7_8_results[
    "final_results_summary"
]
final_conclusion_markdown = phase_7_8_results[
    "final_conclusion_markdown"
]
pre_export_readiness_audit = phase_7_8_results[
    "pre_export_readiness_audit"
]

display(task_recommendations)
display(integrated_model_tradeoff)
display(project_findings)


# Add the measured economics to the final interpretation.
business_finding = pd.DataFrame(
    [
        {
            "topic": "Business case",
            "finding": (
                research_question_7_summary[
                    "marginal_speed_answer"
                ]
            ),
            "meaning": (
                research_question_7_summary[
                    "business_answer"
                ]
            ),
        }
    ]
)
project_findings = pd.concat(
    [project_findings, business_finding],
    ignore_index=True,
)

business_limitations = pd.DataFrame(
    [
        {
            "limitation": (
                "Historical upfront timing"
            ),
            "effect": (
                "Where label-generation or training "
                "wall-clock time was not retained, "
                "lifecycle break-even is reported as "
                "an explicit scenario range rather "
                "than one measured point."
            ),
        },
        {
            "limitation": (
                "Benchmark portability"
            ),
            "effect": (
                "Operational crossover depends on "
                "hardware, threading, batch size, "
                "library version, and whether a "
                "service is already warm."
            ),
        },
    ]
)
project_limitations = pd.concat(
    [
        project_limitations,
        business_limitations,
    ],
    ignore_index=True,
)

final_results_summary[
    "research_question_7"
] = research_question_7_summary
final_results_summary[
    "business_case_status"
] = "complete"
final_results_summary[
    "overall_answer"
] = (
    "Deep learning is not justified merely because "
    "the models can approximate CRR. It earns a "
    "practical role only for repeated in-domain "
    "valuation above the measured operational "
    "crossover and, over the model lifecycle, above "
    "the break-even volume required to repay label "
    "generation and training. Numerical pricing "
    "remains preferable for one-off, changing, or "
    "out-of-domain work."
)

final_conclusion_markdown = (
    final_conclusion_markdown.rstrip()
    + "\n\n"
    + business_case_markdown.strip()
    + "\n"
)

display(business_case_recommendations)


## 31. Final project conclusion

The following conclusion is generated from the validated tables above. It states what the project achieved, which models should be used for which tasks, what the combined model added, where the experiments failed, and which limitations remain.


In [ ]:
display(
    Markdown(final_conclusion_markdown)
)


## 32. Phase 8 — strict final export and validation

The final gate checks the full evidence chain and the final interpretation. It rejects missing hypotheses, missing task recommendations, incomplete out-of-domain coverage, mixed runtime families, missing charts, empty exports, and invalid file hashes.

The final output directory is `artifacts/final_evaluation/final`. It contains the compact tables and write-up inputs needed for submission. The large row-level prediction matrices remain in their Phase 4 and Phase 5–6 directories and are not duplicated.


In [ ]:
from src.evaluation.final_reporting import (
    export_final_project,
    verify_export_manifest,
    write_export_manifest,
)
from src.evaluation.final_validation import (
    assert_phase_7_8_ready,
    build_post_export_readiness_audit,
)

assert_phase_7_8_ready(
    pre_export_readiness_audit
)

final_tables = {
    "task_recommendations": task_recommendations,
    "integrated_model_tradeoff": (
        integrated_model_tradeoff
    ),
    "project_findings": project_findings,
    "project_limitations": project_limitations,
    "hypothesis_decisions": hypothesis_decisions,
    "static_model_metrics": static_model_metrics,
    "static_financial_consistency": (
        static_financial_consistency
    ),
    "exercise_model_metrics": exercise_model_metrics,
    "exercise_boundary_metrics": (
        exercise_boundary_metrics
    ),
    "static_ood_model_summary": (
        static_ood_model_summary
    ),
    "lsm_heldout_pricing": phase_5_6_results[
        "lsm_heldout_pricing"
    ],
    "runtime_comparison": runtime_comparison,
    **business_case_tables,
}

final_export = export_final_project(
    FINAL_OUTPUT_DIR,
    tables=final_tables,
    final_results_summary=final_results_summary,
    final_conclusion_markdown=(
        final_conclusion_markdown
    ),
    chart_paths=final_chart_paths,
    readiness_audit=(
        pre_export_readiness_audit
    ),
)

# Preserve the machine-specific benchmark record and the structured answer to
# Research Question 7 inside the final hashed evidence package.
(
    FINAL_OUTPUT_DIR
    / "runtime_environment.json"
).write_text(
    json.dumps(
        _json_safe(runtime_environment),
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
(
    FINAL_OUTPUT_DIR
    / "research_question_7_summary.json"
).write_text(
    json.dumps(
        _json_safe(
            research_question_7_summary
        ),
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
(
    FINAL_OUTPUT_DIR
    / "research_question_7.md"
).write_text(
    business_case_markdown.strip() + "\n",
    encoding="utf-8",
)

final_export_manifest = write_export_manifest(
    FINAL_OUTPUT_DIR
)
final_export_verification = verify_export_manifest(
    FINAL_OUTPUT_DIR,
    final_export_manifest,
)

required_final_paths = [
    "task_recommendations.csv",
    "integrated_model_tradeoff.csv",
    "project_findings.csv",
    "project_limitations.csv",
    "hypothesis_decisions.csv",
    "hypothesis_decisions.json",
    "static_model_metrics.csv",
    "static_financial_consistency.csv",
    "exercise_model_metrics.csv",
    "exercise_boundary_metrics.csv",
    "static_ood_model_summary.csv",
    "lsm_heldout_pricing.csv",
    "runtime_comparison.csv",
    "runtime_scaling.csv",
    "accuracy_speed_tradeoff.csv",
    "runtime_curves.csv",
    "operational_crossover.csv",
    "upfront_cost_inventory.csv",
    "upfront_cost_scenarios.csv",
    "lifecycle_break_even.csv",
    "business_case_scenarios.csv",
    "business_case_recommendations.csv",
    "business_case_readiness_audit.csv",
    "runtime_environment.json",
    "research_question_7_summary.json",
    "research_question_7.md",
    "final_results_summary.json",
    "final_project_conclusion.md",
    "final_writeup_inputs.md",
    "final_readiness_audit.csv",
    "charts/static_pricing_mae.png",
    "charts/exercise_f1.png",
    "charts/ood_deterioration.png",
    "charts/runtime_comparison.png",
    "charts/lsm_heldout_mae.png",
    "charts/business_runtime_scaling.png",
    "charts/business_speedup_vs_crr.png",
    "charts/business_lifecycle_break_even.png",
    "charts/business_workload_scenarios.png",
]

post_export_readiness_audit = (
    build_post_export_readiness_audit(
        final_export_manifest,
        required_relative_paths=(
            required_final_paths
        ),
        export_verification=(
            final_export_verification
        ),
    )
)

final_readiness_audit = pd.concat(
    [
        pre_export_readiness_audit,
        business_case_readiness_audit,
        post_export_readiness_audit,
    ],
    ignore_index=True,
)

assert_phase_7_8_ready(
    final_readiness_audit
)

final_readiness_audit.to_csv(
    FINAL_OUTPUT_DIR
    / "final_readiness_audit.csv",
    index=False,
)

# Rebuild and verify the manifest after writing the final audit.
final_export_manifest = write_export_manifest(
    FINAL_OUTPUT_DIR
)
final_export_verification = verify_export_manifest(
    FINAL_OUTPUT_DIR,
    final_export_manifest,
)
if (
    final_export_verification.empty
    or not final_export_verification["valid"].all()
):
    raise RuntimeError(
        "Final export files do not match the final manifest."
    )

print("Phases 7–8 readiness gate: PASS")
display(final_readiness_audit)
display(final_export_manifest)


## 33. Project status

A successful final gate means that Notebook 09 has completed the project rather than merely collecting earlier outputs.

The final result is task-specific:

- use the constrained residual model for the most accurate static price;
- use the specialist classifier for the most accurate exercise recommendation;
- use the integrated model when price and exercise information must come from one model;
- use classical Longstaff–Schwartz in the path-based experiment;
- treat all results outside the training range with caution.

The static neural surrogate is valuable because it combines high in-domain accuracy with extremely fast repeated inference. It does not remove the need for the numerical model that generated the labels, and it is not evidence of market-price forecasting ability.


In [ ]:
display(
    pd.Series(
        {
            "status": final_results_summary[
                "status"
            ],
            "universal_preferred_model": (
                final_results_summary[
                    "universal_preferred_model"
                ]
            ),
            "business_case_status": (
                final_results_summary[
                    "business_case_status"
                ]
            ),
            "final_output_directory": str(
                FINAL_OUTPUT_DIR
            ),
            "exported_files": int(
                len(final_export_manifest)
            ),
            "readiness_checks": int(
                len(final_readiness_audit)
            ),
            "all_checks_passed": bool(
                final_readiness_audit[
                    "valid"
                ].all()
            ),
        },
        name="Final project status",
    )
)
